# 05 - Train Pretrained CNNs

This notebook trains the three selected pretrained CNN baselines:

1. EfficientNet-B0
2. DenseNet121
3. ConvNeXt-Tiny

Training strategy:
- Fixed **12 epochs** per model.
- **No early stopping**.
- Two-phase fine-tuning:
  - Epochs 1-3: freeze backbone, train classifier head only.
  - Epochs 4-12: unfreeze full model, fine-tune all layers.
- Save `best_model.pth` by validation macro-F1.
- Save `last_model.pth` at every epoch.
- Evaluate the best checkpoint on the test split.

Use this notebook after PSA+ECA ResNet has completed.

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/backup/Intern/combine_skindiseaseclassifier_devraj')
EXPECTED_PYTHON = '/backup/Intern/combine_skindiseaseclassifier_devraj/.venv/bin/python'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

if sys.executable != EXPECTED_PYTHON:
    raise RuntimeError(
        'Wrong notebook kernel selected. Expected: ' + EXPECTED_PYTHON + '\n'
        'Current: ' + sys.executable + '\n'
        'Please switch kernel to: Combine Skin GPU (.venv)'
    )

Project root: /backup/Intern/combine_skindiseaseclassifier_devraj
Python: /backup/Intern/combine_skindiseaseclassifier_devraj/.venv/bin/python


In [12]:
from datetime import datetime
import json
import subprocess
import time

import pandas as pd
import torch
from torch import nn
from torch.amp import GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.models.pretrained_cnns import (
    DEFAULT_MODEL_SPECS,
    SUPPORTED_MODELS,
    build_pretrained_cnn,
    count_parameters,
    set_backbone_trainable,
)
from src.training.data import build_dataloaders, class_weight_table
from src.training.engine import train_one_epoch, evaluate
from src.training.checkpoint import save_checkpoint, load_checkpoint, save_class_mapping
from src.training.metrics import (
    save_metrics_json,
    save_classification_report,
    save_confusion_matrix,
    save_predictions_csv,
)
from src.utils.seed import set_seed

DATA_DIR = PROJECT_ROOT / 'data' / 'splits'
OUTPUT_ROOT = PROJECT_ROOT / 'training_outputs' / 'pretrained_cnns'
REPORT_MD_DIR = PROJECT_ROOT / 'markdown_reports'

set_seed(42)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('Models:', SUPPORTED_MODELS)

Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
Models: ('efficientnet_b0', 'densenet121', 'convnext_tiny')


## Configuration

This uses 12 fixed epochs and no early stopping.

For all models:
- Epochs 1-3: classifier head only.
- Epochs 4-12: full fine-tuning.

If a model is interrupted, set `RESUME_MODEL_NAME` and `RESUME_RUN_DIR` before rerunning.

In [13]:
REQUIRE_GPU = True
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('CUDA/GPU is not available. Do not train on CPU.')

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0  # keep 0 because this server had /dev/shm DataLoader errors
EPOCHS = 12
FREEZE_EPOCHS = 3
WEIGHT_DECAY = 1e-4
USE_WEIGHTED_SAMPLER = True
PRETRAINED = True

MODELS_TO_TRAIN = ['convnext_tiny']  # training ConvNeXt only for now

# Resume option. Leave both None for normal fresh training.
RESUME_MODEL_NAME = None
RESUME_RUN_DIR = None
# No resume: ConvNeXt will start as a fresh 12-epoch run.

config_base = {
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'epochs': EPOCHS,
    'freeze_epochs': FREEZE_EPOCHS,
    'weight_decay': WEIGHT_DECAY,
    'use_weighted_sampler': USE_WEIGHTED_SAMPLER,
    'pretrained': PRETRAINED,
    'early_stopping': False,
    'patience': None,
    'best_metric': 'val_f1_macro',
    'scheduler': 'CosineAnnealingLR per phase',
}
config_base

{'image_size': 224,
 'batch_size': 32,
 'num_workers': 0,
 'epochs': 12,
 'freeze_epochs': 3,
 'weight_decay': 0.0001,
 'use_weighted_sampler': True,
 'pretrained': True,
 'early_stopping': False,
 'patience': None,
 'best_metric': 'val_f1_macro',
 'scheduler': 'CosineAnnealingLR per phase'}

## Build DataLoaders

Train uses weighted sampling + augmentation. Validation/test stay natural and deterministic.

In [14]:
bundle = build_dataloaders(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampler=USE_WEIGHTED_SAMPLER,
)

num_classes = len(bundle.class_to_idx)
metric_labels = list(range(num_classes))
class_names = [bundle.idx_to_class[idx] for idx in metric_labels]

print('Classes:', num_classes)
print('Train:', len(bundle.train_dataset))
print('Val:', len(bundle.val_dataset))
print('Test:', len(bundle.test_dataset))

weights_df = class_weight_table(bundle.idx_to_class, bundle.class_counts, bundle.class_weights)
weights_df

Classes: 14
Train: 25576
Val: 5478
Test: 5474


,class_idx,class_name,train_count,class_weight
0,0,acne_vulgaris,1727,1.057821
1,1,atopic_dermatitis,1376,1.327658
2,2,basal_cell_carcinoma,4247,0.430152
3,3,contact_dermatitis,1062,1.720205
4,4,drug_eruptions,1115,1.638437
5,5,folliculitis,825,2.214372
6,6,fungal_nail_infections,844,2.164523
7,7,lupus_related_skin_lesions,900,2.029841
8,8,melanoma,6367,0.286926
9,9,plaque_psoriasis,2342,0.780041


In [15]:
images, labels = next(iter(bundle.train_loader))
print('Batch image shape:', images.shape)
print('Batch label shape:', labels.shape)
pd.Series([bundle.idx_to_class[int(label)] for label in labels]).value_counts()

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])


atopic_dermatitis             6
acne_vulgaris                 4
contact_dermatitis            4
warts                         3
fungal_nail_infections        2
plaque_psoriasis              2
basal_cell_carcinoma          2
melanoma                      2
folliculitis                  2
drug_eruptions                2
lupus_related_skin_lesions    1
vitiligo                      1
tinea_corporis                1
Name: count, dtype: int64

## Helper Functions

In [16]:
def make_run_dir(model_name):
    run_name = f'{model_name}_fixed_12ep_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    run_dir = OUTPUT_ROOT / model_name / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_name, run_dir


def current_phase(epoch):
    return 'head_only' if epoch <= FREEZE_EPOCHS else 'full_finetune'


def configure_phase(model, model_name, phase, spec):
    if phase == 'head_only':
        set_backbone_trainable(model, model_name, trainable=False)
        lr = spec.phase1_lr
        phase_epochs = FREEZE_EPOCHS
    else:
        set_backbone_trainable(model, model_name, trainable=True)
        lr = spec.phase2_lr
        phase_epochs = EPOCHS - FREEZE_EPOCHS

    optimizer = AdamW((p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=max(phase_epochs, 1))
    return optimizer, scheduler, lr


def save_test_outputs(model, criterion, run_dir, run_name, model_name, config, best_checkpoint):
    checkpoint = load_checkpoint(best_checkpoint, model, map_location=device)
    model.to(device)
    model.eval()
    best_epoch = int(checkpoint['epoch'])

    test_metrics = evaluate(
        model=model,
        loader=bundle.test_loader,
        criterion=criterion,
        device=device,
        return_predictions=True,
        metric_labels=metric_labels,
    )
    y_true = test_metrics.pop('y_true')
    y_pred = test_metrics.pop('y_pred')
    y_prob = test_metrics.pop('y_prob')

    result_payload = {
        'model': model_name,
        'run_name': run_name,
        'best_epoch': best_epoch,
        'best_checkpoint': str(best_checkpoint),
        'test_metrics': test_metrics,
        'config': config,
    }
    save_metrics_json(result_payload, run_dir / 'metrics.json')
    save_classification_report(y_true, y_pred, class_names, run_dir / 'classification_report.txt')
    save_confusion_matrix(y_true, y_pred, class_names, run_dir / 'confusion_matrix.png', normalize=False)
    save_confusion_matrix(y_true, y_pred, class_names, run_dir / 'confusion_matrix_normalized.png', normalize=True)

    image_paths = [sample[0] for sample in bundle.test_dataset.samples]
    save_predictions_csv(image_paths, y_true, y_pred, y_prob, bundle.idx_to_class, run_dir / 'predictions_test.csv')
    return result_payload

## Train One Model

This function trains one model for fixed 12 epochs. It saves best and last checkpoints.

In [17]:
def train_one_pretrained_model(model_name, resume_run_dir=None):
    spec = DEFAULT_MODEL_SPECS[model_name]

    if resume_run_dir is None:
        run_name, run_dir = make_run_dir(model_name)
        config = {
            **config_base,
            'model': model_name,
            'run_name': run_name,
            'phase1_lr': spec.phase1_lr,
            'phase2_lr': spec.phase2_lr,
            'resumed': False,
        }
        (run_dir / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
        start_epoch = 1
        log_rows = []
        best_val_f1 = -1.0
        best_epoch = None
    else:
        run_dir = Path(resume_run_dir)
        run_name = run_dir.name
        config = json.loads((run_dir / 'config.json').read_text(encoding='utf-8'))
        config['resumed'] = True
        config['resume_from'] = str(run_dir / 'last_model.pth')
        (run_dir / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')

        log_path = run_dir / 'training_log.csv'
        old_log = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
        log_rows = old_log.to_dict('records') if len(old_log) else []
        start_epoch = int(old_log['epoch'].max()) + 1 if len(old_log) else 1
        if len(old_log):
            best_idx = old_log['val_f1_macro'].idxmax()
            best_val_f1 = float(old_log.loc[best_idx, 'val_f1_macro'])
            best_epoch = int(old_log.loc[best_idx, 'epoch'])
        else:
            best_val_f1 = -1.0
            best_epoch = None

    print('\n' + '=' * 72)
    print(f'Training model: {model_name}')
    print(f'Run folder    : {run_dir}')
    print('=' * 72)

    model = build_pretrained_cnn(model_name, num_classes=num_classes, pretrained=PRETRAINED).to(device)
    total_params, _ = count_parameters(model)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler('cuda') if device.type == 'cuda' else None

    save_class_mapping(bundle.class_to_idx, run_dir / 'class_to_idx.json')
    weights_df.to_csv(run_dir / 'class_weights_used.csv', index=False)

    if resume_run_dir is not None:
        last_checkpoint = run_dir / 'last_model.pth'
        if last_checkpoint.exists():
            checkpoint = torch.load(last_checkpoint, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f'Resuming from completed epoch {checkpoint["epoch"]}; next epoch {start_epoch}')
        else:
            print('Resume folder has no last_model.pth yet. Restarting this model from epoch 1 in the same folder.')

    (run_dir / 'model_summary.txt').write_text(
        f'Model: {model_name}\nTotal params: {total_params:,}\nClasses: {num_classes}\nTwo-phase fine-tuning: {FREEZE_EPOCHS}+{EPOCHS-FREEZE_EPOCHS}\n',
        encoding='utf-8',
    )

    optimizer = None
    scheduler = None
    active_phase = None
    start_time = time.time()

    if start_epoch > EPOCHS:
        print(f'{model_name} already completed {EPOCHS} epochs.')
    else:
        for epoch in range(start_epoch, EPOCHS + 1):
            phase = current_phase(epoch)
            if phase != active_phase:
                optimizer, scheduler, phase_lr = configure_phase(model, model_name, phase, spec)
                active_phase = phase
                print(f'\nSwitched to phase: {phase} | lr={phase_lr}')

            print(f'\n{model_name} - Epoch {epoch}/{EPOCHS} ({phase})')
            train_metrics = train_one_epoch(
                model=model,
                loader=bundle.train_loader,
                criterion=criterion,
                optimizer=optimizer,
                device=device,
                scaler=scaler,
                metric_labels=metric_labels,
            )
            val_metrics = evaluate(
                model=model,
                loader=bundle.val_loader,
                criterion=criterion,
                device=device,
                return_predictions=False,
                metric_labels=metric_labels,
            )

            val_f1 = float(val_metrics['f1_macro'])
            improved = val_f1 > best_val_f1
            scheduler.step()

            _, trainable_params = count_parameters(model)
            row = {
                'epoch': epoch,
                'phase': phase,
                'lr': optimizer.param_groups[0]['lr'],
                'trainable_params': trainable_params,
                **{f'train_{k}': v for k, v in train_metrics.items()},
                **{f'val_{k}': v for k, v in val_metrics.items()},
                'improved': improved,
            }
            log_rows.append(row)
            pd.DataFrame(log_rows).to_csv(run_dir / 'training_log.csv', index=False)

            print(
                f"train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['f1_macro']:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['f1_macro']:.4f} "
                f"val_acc={val_metrics['accuracy']:.4f} lr={optimizer.param_groups[0]['lr']:.2e}"
            )

            save_checkpoint(
                run_dir / 'last_model.pth',
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                metrics={'train': train_metrics, 'val': val_metrics},
                class_to_idx=bundle.class_to_idx,
                extra=config,
            )

            if improved:
                best_val_f1 = val_f1
                best_epoch = epoch
                save_checkpoint(
                    run_dir / 'best_model.pth',
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    metrics={'train': train_metrics, 'val': val_metrics},
                    class_to_idx=bundle.class_to_idx,
                    extra=config,
                )
                print(f'Saved new best checkpoint at epoch {epoch} with val macro-F1={best_val_f1:.4f}')

    elapsed_minutes = (time.time() - start_time) / 60
    print(f'Finished {model_name} in {elapsed_minutes:.2f} minutes. Best epoch: {best_epoch}')

    result_payload = save_test_outputs(
        model=model,
        criterion=criterion,
        run_dir=run_dir,
        run_name=run_name,
        model_name=model_name,
        config=config,
        best_checkpoint=run_dir / 'best_model.pth',
    )
    print('Test metrics:')
    print(json.dumps(result_payload['test_metrics'], indent=2))
    return result_payload

## Train Models

Run this cell to train all three models one by one.

If you want to test only one model first, change `MODELS_TO_TRAIN` in the configuration cell.

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', device)

all_results = []
for model_name in MODELS_TO_TRAIN:
    resume_dir = RESUME_RUN_DIR if RESUME_MODEL_NAME == model_name else None
    result = train_one_pretrained_model(model_name, resume_run_dir=resume_dir)
    all_results.append(result)

all_results

Training device: cuda

Training model: convnext_tiny
Run folder    : /backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/pretrained_cnns/convnext_tiny/convnext_tiny_fixed_12ep_20260701_070637
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /home/rohan/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:05<00:00, 19.8MB/s] 



Switched to phase: head_only | lr=0.001

convnext_tiny - Epoch 1/12 (head_only)


train_loss=1.2242 train_f1=0.5937 | val_loss=0.9574 val_f1=0.6411 val_acc=0.6871 lr=7.50e-04
Saved new best checkpoint at epoch 1 with val macro-F1=0.6411

convnext_tiny - Epoch 2/12 (head_only)


train_loss=0.9624 train_f1=0.6811 | val_loss=0.9031 val_f1=0.6497 val_acc=0.6984 lr=2.50e-04
Saved new best checkpoint at epoch 2 with val macro-F1=0.6497

convnext_tiny - Epoch 3/12 (head_only)


...

train_loss=0.0571 train_f1=0.9866 | val_loss=0.5455 val_f1=0.8168 val_acc=0.8474 lr=5.85e-06
Saved new best checkpoint at epoch 10 with val macro-F1=0.8168

convnext_tiny - Epoch 11/12 (full_finetune)


train_loss=0.0475 train_f1=0.9885 | val_loss=0.5428 val_f1=0.8216 val_acc=0.8518 lr=1.51e-06
Saved new best checkpoint at epoch 11 with val macro-F1=0.8216

convnext_tiny - Epoch 12/12 (full_finetune)


train_loss=0.0429 train_f1=0.9900 | val_loss=0.5326 val_f1=0.8219 val_acc=0.8527 lr=0.00e+00
Saved new best checkpoint at epoch 12 with val macro-F1=0.8219
Finished convnext_tiny in 205.34 minutes. Best epoch: 12


Test metrics:
{
  "accuracy": 0.8584216295213738,
  "precision_macro": 0.8306795412612351,
  "recall_macro": 0.8326050572084072,
  "f1_macro": 0.8309382207764523,
  "f1_weighted": 0.8582943480211983,
  "loss": 0.5164879600817401
}


[{'model': 'convnext_tiny',
  'run_name': 'convnext_tiny_fixed_12ep_20260701_070637',
  'best_epoch': 12,
  'best_checkpoint': '/backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/pretrained_cnns/convnext_tiny/convnext_tiny_fixed_12ep_20260701_070637/best_model.pth',
  'test_metrics': {'accuracy': 0.8584216295213738,
   'precision_macro': 0.8306795412612351,
   'recall_macro': 0.8326050572084072,
   'f1_macro': 0.8309382207764523,
   'f1_weighted': 0.8582943480211983,
   'loss': 0.5164879600817401},
  'config': {'image_size': 224,
   'batch_size': 32,
   'num_workers': 0,
   'epochs': 12,
   'freeze_epochs': 3,
   'weight_decay': 0.0001,
   'use_weighted_sampler': True,
   'pretrained': True,
   'early_stopping': False,
   'patience': None,
   'best_metric': 'val_f1_macro',
   'scheduler': 'CosineAnnealingLR per phase',
   'model': 'convnext_tiny',
   'run_name': 'convnext_tiny_fixed_12ep_20260701_070637',
   'phase1_lr': 0.001,
   'phase2_lr': 5e-05,
   'resumed': Fal

## Save Notebook-Level Summary

In [19]:
summary_rows = []
for result in all_results:
    row = {
        'model': result['model'],
        'run_name': result['run_name'],
        'best_epoch': result['best_epoch'],
        'best_checkpoint': result['best_checkpoint'],
        **result['test_metrics'],
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_dir = PROJECT_ROOT / 'reports' / 'pretrained_cnns'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_csv = summary_dir / 'pretrained_cnn_test_summary.csv'
summary_df.to_csv(summary_csv, index=False)
summary_df

,model,run_name,best_epoch,best_checkpoint,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,loss
0,convnext_tiny,convnext_tiny_fixed_12ep_20260701_070637,12,/backup/Intern/combine_skindiseaseclassifier_d...,0.858422,0.83068,0.832605,0.830938,0.858294,0.516488


In [20]:
md_path = REPORT_MD_DIR / 'pretrained_cnn_12epoch_summary.md'
with md_path.open('w', encoding='utf-8') as f:
    f.write('# Pretrained CNN 12-Epoch Summary\n\n')
    f.write('Models trained with fixed 12 epochs, no early stopping, and two-phase fine-tuning.\n\n')
    f.write('Phase 1: epochs 1-3, frozen backbone, classifier head only.\n\n')
    f.write('Phase 2: epochs 4-12, full model fine-tuning.\n\n')
    f.write('| Model | Best Epoch | Test Accuracy | Test Macro-F1 | Test Recall Macro | Test Precision Macro |\n')
    f.write('|---|---:|---:|---:|---:|---:|\n')
    for _, row in summary_df.iterrows():
        f.write(
            f"| {row['model']} | {int(row['best_epoch'])} | "
            f"{row['accuracy']:.4f} | {row['f1_macro']:.4f} | "
            f"{row['recall_macro']:.4f} | {row['precision_macro']:.4f} |\n"
        )
    f.write('\nSummary CSV: `reports/pretrained_cnns/pretrained_cnn_test_summary.csv`\n')
md_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/markdown_reports/pretrained_cnn_12epoch_summary.md')

## Update Project Structure

In [21]:
subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'update_project_structure.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

Updated: /backup/Intern/combine_skindiseaseclassifier_devraj/PROJECT_STRUCTURE.txt


CompletedProcess(args=['python3', '/backup/Intern/combine_skindiseaseclassifier_devraj/scripts/update_project_structure.py'], returncode=0)